# Cryptocurrency Volatility Prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
df=pd.read_csv(r'dataset.csv')
df.head()

In [ ]:
# Basic preprocessing
df.columns=df.columns.str.lower().str.strip()
df=df.drop_duplicates()
df=df.fillna(method='ffill').fillna(method='bfill')
# Feature engineering if OHLC exists
if set(['open','high','low','close']).issubset(df.columns):
    df['returns']=df['close'].pct_change()
    df['volatility']=df['returns'].rolling(7).std()
    df['ma7']=df['close'].rolling(7).mean()
    df['ma30']=df['close'].rolling(30).mean()
if set(['volume','marketcap']).issubset(df.columns):
    df['liq_ratio']=df['volume']/(df['marketcap']+1)
df=df.dropna()
target='volatility'
features=[c for c in df.select_dtypes(include='number').columns if c!=target]
X=df[features]; y=df[target]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,shuffle=False)
model=RandomForestRegressor(n_estimators=200,random_state=42)
model.fit(X_train,y_train)
pred=model.predict(X_test)
print('MAE',mean_absolute_error(y_test,pred))
print('RMSE',mean_squared_error(y_test,pred)**0.5)
print('R2',r2_score(y_test,pred))
